# Limpieza y Analisis a detalle de la base de datos


In [1]:
import sys
import pandas as pd
sys.path.append("..")  # subir un nivel

from scripts import bases as b, init_notebook as init, limpieza as lim
ctx = init.init_notebook()


df_pacientes = ctx["df_pacientes"]
df_pacientes = lim.limpiar_dataset_pro(df_pacientes)
traslados = ctx["traslados"]
hosp_coords = ctx["hosp_coords"]
municipios = ctx["municipios"]
municipios_amba = ctx["municipios_amba"]

Cargando datos principales...
Pacientes y traslados cargados.
Cargando datos geográficos...
Datos geográficos cargados.


In [ ]:
# exploracion
# lim.explorar_valores(df_pacientes)

In [ ]:
# debug / calidad de datos
lim.detectar_problemas(df_pacientes)
lim.check_coherencia(df_pacientes)

In [ ]:
# check
lim.check_post_limpieza(df_pacientes)

## 1. Funcionamiento de la Red General (AMBA)

## 2. Trayectorias de Paciente

### 2.1 Cantidad de traslados para cada persona (promedio y desvío, junto a distribución)

In [ ]:
# Distribución de traslados por paciente
conteo_tras_paciente, stats_tras_paciente = bases.distribucion_traslados_paciente(traslados, col_id="Id", valores=[1, 2, 3, 4, 5, 6, 7], graficar=True)

In [ ]:
## que paso con estos 3?
## reconstruir estos 3

# incluir los ingresos como porcentaje. escala logaritmica

# PROBABILIDAD DE TENER UN TRASLADO MAS DADO LA CANTIDAD DE TRASLADOS QUE TUVISTE

# Cuantos de esos tambien tienen como lo que tuviste

In [ ]:
#bases.revisar_dias_negativos(traslados, max_pacientes=10)

In [ ]:
# # pacientes con error
# errores = traslados[traslados["error_fecha"]]
# print("Pacientes con errores de fechas:", errores["Id"].unique())

# # historial completo de un paciente con error
# # usar el DataFrame de traslados que tiene las columnas calculadas
# pid = errores["Id"].iloc[1]

# historial = traslados[traslados["Id"] == pid].sort_values("Fecha inicio")

# display(historial[[
#     "Nombre Hospital",
#     "Fecha inicio",
#     "Fecha egreso",
#     "Hospital siguiente",
#     "Fecha ingreso siguiente",
#     "dias_entre_hospitales",
#     "error_fecha",
#     "gravedad_error"
# ]])

In [ ]:
# pacientes con >=3 traslados
traslados_muchos = bases.pacientes_con_muchos_traslados(traslados, minimo=3)

# imprimir recorridos
# bases.imprimir_recorridos_pacientes(
#     traslados_muchos,
#     col_id="Id",
#     col_origen="Nombre Hospital",
#     col_destino="Hospital siguiente"
# )

In [ ]:
def detectar_rebotes(df, umbral_dias=1):
    
    rebotes = df[
        (df["dias_entre_hospitales"] >= 0) &
        (df["dias_entre_hospitales"] <= umbral_dias)
    ]
    
    print("Cantidad de rebotes:", len(rebotes))
    
    display(
        rebotes[[
            "Id",
            "Nombre Hospital",
            "Hospital siguiente",
            "dias_entre_hospitales"
        ]].head(10)
    )
    
    return rebotes

rebotes=detectar_rebotes(traslados)
df_pacientes.head(5)

In [ ]:
#bases.graficar_estado_paciente_debug(traslados_muchos.head(5))

In [ ]:
# # ------------------------------
# # EJECUCIÓN EN EL NOTEBOOK
# # ------------------------------
# # Mostrar tablas por paciente
# bases.mostrar_recorridos_estado(traslados_muchos.head(9))

# # Sankey de flujo
# bases.sankey_pacientes(traslados_muchos)

### 2.2 Tiempo dentro del sistema por persona

In [ ]:
# Tiempo en el sistema por persona
#tiempo_sis, limite_tiempo = bases.tiempo_total_paciente(df_pacientes, col_id="Id", col_dias="Duracion días", max_dias=100, quantile_outlier=0.95, graficar=True)

In [ ]:
# print("n pacientes:", len(tiempo_sis))
# print("min:", tiempo_sis.min())
# print("max:", tiempo_sis.max())
# print("valores:", tiempo_sis.value_counts().head(10))

In [ ]:
# un grafico grande y te vas quedando con info
# ir bajandolo a temas

## 3. Análisis Descriptivo por Hospital

### 3.1 Traslados por hospital

In [ ]:
# # Traslados OUT (Origen)
# traslados_out = bases.traslados_por_hospital(traslados, col_hospital="Nombre Hospital", graficar=False)

# # Traslados IN (Destino)
# traslados_in = bases.traslados_por_hospital(traslados, col_hospital="Hospital siguiente", graficar=False)

# tabla_hospitales = pd.DataFrame({
#     "traslados_out": traslados_out,
#     "traslados_in": traslados_in,
# }).fillna(0)

# print("Top 10 hospitales que derivan más pacientes:")
# display(tabla_hospitales.sort_values(by="traslados_out", ascending=False).head(10))

# print("\nTop 10 hospitales que reciben más pacientes:")
# display(tabla_hospitales.sort_values(by="traslados_in", ascending=False).head(10))

# # Graficamos los IN
# tabla_hospitales["traslados_in"].sort_values(ascending=False).head(10).plot(kind="bar", color="skyblue", figsize=(10,5))
# plt.title("Top 10 Hospitales por Ingresos (Traslados IN)")
# plt.ylabel("Cantidad de traslados recibidos")
# plt.xlabel("Hospital")
# plt.xticks(rotation=45, ha='right')
# plt.show()

In [ ]:
# ESTO ES STRENGTH O FUERZA
# DEGREE: # de hospitales con los que me conecto

# fuerza/grado = numero promedio de traslados que recibo de un hospital con el que estoy conectado

### 3.2 Tiempo promedio que pasa una persona dentro del hospital

In [ ]:
# # Tiempo promedio por hospital
# tiempo_prom_hosp = bases.tiempo_promedio_por_hospital(df_pacientes, col_hospital="Nombre Hospital", col_dias="Duracion días", quantile_outlier=0.95, graficar=False)

# display(tiempo_prom_hosp.head(10))

# tiempo_prom_hosp.head(15).plot(kind="bar", color="orange", figsize=(10,5))
# plt.title("Top 15 Hospitales por Tiempo Promedio de Estadía")
# plt.ylabel("Días promedio")
# plt.xlabel("Hospital")
# plt.xticks(rotation=45, ha='right')
# plt.show()

In [ ]:
# agregar los valores a las barras

### 3.3 Cantidad de muertos por hospital

In [ ]:
# muertes_hosp = bases.muertes_por_hospital(
#     df_pacientes,
#     col_hospital="Nombre Hospital",
#     col_muerte="murio",
#     graficar=False
# )

# display(muertes_hosp.head(10))

# ax = muertes_hosp.head(15).plot(kind="bar", color="red", figsize=(10,5))

# # agregar numerito arriba de cada barra
# for p in ax.patches:
#     height = p.get_height()
#     ax.text(
#         p.get_x() + p.get_width()/2,
#         height,
#         str(int(height)),
#         ha="center",
#         va="bottom",
#         fontsize=9
#     )

# plt.title("Top 15 Hospitales por Cantidad de Fallecidos")
# plt.ylabel("Cantidad de fallecidos")
# plt.xlabel("Hospital")
# plt.xticks(rotation=45, ha='right')
# plt.show()

In [ ]:
## entrar a la red o volver a casa o muerte
## version: optimista


# ver que rol tienen estos indicadores en la caminata

In [ ]:
## ver los cambios de estado de las personas
## cuanta gente pasa de estado general a critico, intermedio a general, etc etc

## traslados entre hospitales  y traslados entre niveles
# la gente mejora o empeora en el hospital? cuantos?

## 4. Análisis Combinado

### 4.1 Cantidad de personas con distintos niveles de riesgo social y estados (crítico, intermedio general)

In [ ]:
# # Tabla cruzada de Nivel de riesgo social y Estado al ingreso
# # Tabla cruzada de Nivel de riesgo social y Estado al ingreso
# if "Nivel riesgo social" in df_pacientes.columns and "Estado al ingreso" in df_pacientes.columns:

#     tabla_riesgo_estado = pd.crosstab(
#         df_pacientes["Nivel riesgo social"],
#         df_pacientes["Estado al ingreso"],
#         margins=True,
#         margins_name="Total"
#     )

#     display(tabla_riesgo_estado)

#     tabla_sin_totales = pd.crosstab(
#         df_pacientes["Nivel riesgo social"],
#         df_pacientes["Estado al ingreso"]
#     )

#     tabla_sin_totales.plot(
#         kind="bar",
#         stacked=False,
#         figsize=(10,6),
#         colormap="viridis"
#     )

#     plt.title("Estado al ingreso según Nivel de Riesgo Social")
#     plt.ylabel("Cantidad de pacientes")
#     plt.xlabel("Nivel de Riesgo")
#     plt.xticks(rotation=0)
#     plt.legend(title="Estado al ingreso")
#     plt.tight_layout()
#     plt.show()

# else:
#     print("Las columnas necesarias para este análisis no están disponibles.")
#     print("Columnas disponibles:", df_pacientes.columns.tolist())